In [0]:
# Notebook parameters

params = {
    "proj_dir": "/Volumes/7_outgoing/pharos/imaging/pharos_20260619/",
    "output_path": "/Volumes/7_outgoing/pharos/imaging/pharos_20260619_metadata.parquet"
}

# create text widgets
for k in params.keys():
    dbutils.widgets.text(k, params[k], "")

# fetch values
for k in params.keys():
    params[k] = dbutils.widgets.get(k)
    print(k, ":", params[k])

In [0]:
from pyspark.sql.functions import lit, col, from_json, schema_of_json, regexp_replace
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
import pandas as pd
import pydicom
from pydicom.errors import InvalidDicomError
from datetime import datetime
from typing import Iterator
from glob import glob
import os

In [0]:
scan_dirs = glob(f"{params['proj_dir']}/*/*/")
from pyspark.sql import Row

scan_df = spark.createDataFrame([Row(scan_dir=sd) for sd in scan_dirs])

def file_01_exists(scan_dir):
    if os.path.exists(os.path.join(scan_dir, "000001.dcm")):
        return "dbfs:"+os.path.join(scan_dir, "000001.dcm")
    else:
        dcm_file = glob(os.path.join(scan_dir, "*.dcm"))[0]
        return "dbfs:"+os.path.join(scan_dir, dcm_file)


from pyspark.sql.functions import udf
from pyspark.sql.types import BooleanType, StringType
from pyspark.sql.functions import expr

file_exists_udf = udf(file_01_exists, StringType())

scan_df = scan_df.withColumn("dcm_file", file_exists_udf("scan_dir"))
scan_df = scan_df.withColumnRenamed("scan_dir", "scan_dir_path")
scan_df = scan_df.withColumn("dcm_file", expr(f"replace(dcm_file, 'dbfs:', '')"))

from pyspark.sql.functions import expr

scan_df = scan_df.withColumn("scan_dir", expr(f"replace(scan_dir_path, '{params['proj_dir']}', '')"))



display(scan_df.limit(1000))

In [0]:




# Output schema
schema = StructType([
    StructField("scan_dir_path", StringType(), True),
    StructField("dcm_file", StringType(), True),
    StructField("scan_dir", StringType(), True),
    StructField("accession_number", StringType(), True),
    StructField("study_datetime", TimestampType(), True),
    StructField("modality", StringType(), True),
    StructField("study_description", StringType(), True),
])

# mapInPandas function
def extract_dicom_tags(iterator: Iterator[pd.DataFrame]) -> Iterator[pd.DataFrame]:
    for pdf in iterator:
        results = []

        for _, row in pdf.iterrows():
            full_path = row["scan_dir_path"]
            dcm_path = row["dcm_file"]
            subdir = row["scan_dir"]

            try:
                ds = pydicom.dcmread(
                    dcm_path,
                    stop_before_pixels=True,
                    force=True
                )

                accession_number = str(ds.get("AccessionNumber", "")) or None
                modality = str(ds.get("Modality", "")) or None
                study_description = str(ds.get("StudyDescription", "")) or None

                # Build Timestamp from StudyDate + StudyTime
                study_datetime = None
                study_date = str(ds.get("StudyDate", "")).strip()
                study_time = str(ds.get("StudyTime", "")).split(".")[0].strip()

                if study_date:
                    try:
                        if study_time:
                            dt_str = study_date + study_time.ljust(6, "0")
                            study_datetime = datetime.strptime(
                                dt_str, "%Y%m%d%H%M%S"
                            )
                        else:
                            study_datetime = datetime.strptime(
                                study_date, "%Y%m%d"
                            )
                    except Exception:
                        study_datetime = None

                results.append({
                    "scan_dir_path": full_path,
                    "dcm_file": dcm_path,
                    "scan_dir": subdir,
                    "accession_number": accession_number,
                    "study_datetime": study_datetime,
                    "modality": modality,
                    "study_description": study_description,
                })

            except (InvalidDicomError, FileNotFoundError, Exception):
                results.append({
                    "scan_dir_path": full_path,
                    "dcm_file": dcm_path,
                    "scan_dir": subdir,
                    "accession_number": None,
                    "study_datetime": None,
                    "modality": None,
                    "study_description": None,
                })

        yield pd.DataFrame(results)

# Execute extraction
dicom_tags_df = scan_df.mapInPandas(
    extract_dicom_tags,
    schema=schema
)

output_path = params["output_path"]
dicom_tags_df.write.mode("overwrite").parquet(output_path)

In [0]:
display(dicom_tags_df.filter("modality = 'US'").select("accession_number").distinct().count())